# SmartSeniors — Transcription d'appels + Biblio « Emma conseillère »

Ce notebook fait deux choses :

1. **Speech-to-text** : transcrit un appel `.m4a` (conseiller **Alex** ↔ **famille**) en texte, avec séparation des locuteurs (optionnel).
2. **Biblio** : à partir du transcript, extrait une **fiche conseillère** structurée (questions de découverte, données → champs de lead, objections, phrases d'empathie, vocabulaire, transitions) via Claude, puis synthétise un **playbook** pour affiner le prompt d'Emma.

**Avant de lancer** : `Exécution → Modifier le type d'exécution → T4 GPU`. Puis exécute les cellules dans l'ordre.

> ⚠️ **RGPD** : un appel réel contient des données personnelles et de santé d'une vraie famille. Garde les transcripts bruts **hors du repo** (cf. `knowledge/transcripts/raw/`, ignoré par git). N'archive que des versions **anonymisées** et les fiches.

## Partie A — Transcription (Whisper)

In [ ]:
# Vérifie le GPU
!nvidia-smi -L 2>/dev/null || echo "⚠️ Pas de GPU. Exécution → Modifier le type d'exécution → T4 GPU, puis relance."

In [ ]:
!pip -q install -U "faster-whisper>=1.1.0"

In [ ]:
# Charge l'audio. Option 1 : upload direct.
from google.colab import files
print("Choisis ton .m4a (l'appel Alex ↔ famille)…")
up = files.upload()
AUDIO_PATH = next(iter(up))
print("Fichier :", AUDIO_PATH)

# Option 2 (gros fichiers) : monter Google Drive et pointer le chemin
# from google.colab import drive; drive.mount("/content/drive")
# AUDIO_PATH = "/content/drive/MyDrive/appel_alex.m4a"

In [ ]:
from faster_whisper import WhisperModel
import torch

device  = "cuda" if torch.cuda.is_available() else "cpu"
compute = "float16" if device == "cuda" else "int8"
print(f"Modèle large-v3 sur {device} ({compute})…")
model = WhisperModel("large-v3", device=device, compute_type=compute)

segments, info = model.transcribe(AUDIO_PATH, language="fr", vad_filter=True, beam_size=5)
segments = list(segments)
print(f"Durée : {info.duration:.0f}s — {len(segments)} segments")

def hms(s):
    s = int(s); return f"{s//3600:02d}:{(s%3600)//60:02d}:{s%60:02d}"

plain = " ".join(seg.text.strip() for seg in segments)
timed = "\n".join(f"[{hms(seg.start)}] {seg.text.strip()}" for seg in segments)

open("transcript_brut.txt", "w").write(plain)
open("transcript_horodate.txt", "w").write(timed)
print(timed[:1500])

## Partie A bis — Séparation des locuteurs (optionnel : Alex / Famille)

Donne des transcripts en dialogue (`Alex:` / `Famille:`), beaucoup plus utiles pour la biblio.
Nécessite un token HuggingFace **et** d'accepter les conditions de :
`huggingface.co/pyannote/speaker-diarization-3.1` **et** `huggingface.co/pyannote/segmentation-3.0`.

> Si tu sautes cette partie, la biblio (Partie B) fonctionne très bien sur le transcript brut.

In [ ]:
!pip -q install -U "pyannote.audio>=3.1"

import os, torch
from pyannote.audio import Pipeline

HF_TOKEN = os.environ.get("HF_TOKEN") or input("Token HuggingFace (hf_…) : ").strip()
dia_pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", use_auth_token=HF_TOKEN)
dia_pipeline.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

diarization = dia_pipeline(AUDIO_PATH)
turns = [(t.start, t.end, spk) for t, _, spk in diarization.itertracks(yield_label=True)]
print(f"{len(turns)} tours de parole — locuteurs : {sorted(set(s for *_, s in turns))}")

In [ ]:
# Associe chaque segment Whisper au locuteur dominant (recouvrement max), puis nomme les rôles.
def speaker_for(start, end):
    best, best_ov = None, 0.0
    for ts, te, spk in turns:
        ov = max(0.0, min(end, te) - max(start, ts))
        if ov > best_ov: best, best_ov = spk, ov
    return best or "SPEAKER_?"

# Heuristique : le 1er à parler = Alex (le conseiller). ⚠️ Vérifie le rendu et inverse si besoin :
#   ROLE = {first_spk: "Famille"}
first_spk = speaker_for(segments[0].start, segments[0].end)
ROLE = {first_spk: "Alex"}
def role(spk): return ROLE.get(spk, "Famille")

lignes, cur, buf = [], None, []
for seg in segments:
    spk = role(speaker_for(seg.start, seg.end))
    if spk != cur and buf:
        lignes.append(f"{cur}: {' '.join(buf)}"); buf = []
    cur = spk; buf.append(seg.text.strip())
if buf: lignes.append(f"{cur}: {' '.join(buf)}")

transcript_dialogue = "\n".join(lignes)
open("transcript_dialogue.txt", "w").write(transcript_dialogue)
print(transcript_dialogue[:2000])
files.download("transcript_dialogue.txt")

## Partie B — Biblio : extraire la « fiche conseillère » (Claude + tool use)

On utilise **`claude-opus-4-8`** avec **tool use forcé** : le schéma de la fiche est généré depuis Pydantic, la réponse est garantie structurée puis revalidée.
**Prompt caching** : le préfixe `system + outil` est mis en cache (réutilisé d'un appel à l'autre quand tu traites plusieurs transcripts), et le transcript aussi (réutilisé si tu relances l'extraction). Regarde `cache_read` grimper.

In [ ]:
!pip -q install -U anthropic pydantic

In [ ]:
import os
from getpass import getpass
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Clé API Anthropic (sk-ant-…) : ")

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal

class QuestionDecouverte(BaseModel):
    ordre: int = Field(description="Position dans l'appel (1 = première question)")
    question: str = Field(description="La question telle que posée, reformulée proprement")
    intention: str = Field(description="Ce que le conseiller cherche à apprendre")
    etape: str = Field(description="contexte_emotionnel | profil | autonomie | localisation | budget | delai | criteres")
    champ_lead: Optional[str] = Field(default=None, description="Champ de lead renseigné, si applicable")

class PointDonnee(BaseModel):
    champ_lead: str = Field(description="Champ de lead, ex: prenom_proche, ville_recherche, budget_mensuel, delai")
    valeur: str = Field(description="Valeur captée dans l'appel")
    verbatim: str = Field(description="Extrait du transcript (ANONYMISÉ) d'où vient l'info")
    etape: Literal["identite", "solution"] = Field(description="identite = création du lead ; solution = critères de matching")

class Objection(BaseModel):
    objection: str = Field(description="Ce que la famille exprime comme frein")
    reponse_conseiller: str = Field(description="Comment le conseiller y répond")
    technique: str = Field(description="Nom de la technique (réassurance financière, dédramatisation, etc.)")

class PhraseEmpathie(BaseModel):
    phrase: str
    contexte: str = Field(description="Quand cette phrase est employée")

class Transition(BaseModel):
    de: str
    vers: str
    formulation: str = Field(description="La phrase qui fait passer d'une étape à l'autre")

class FicheConseillere(BaseModel):
    resume_appel: str
    questions_decouverte: List[QuestionDecouverte]
    points_donnees: List[PointDonnee]
    objections: List[Objection]
    phrases_empathie: List[PhraseEmpathie]
    vocabulaire: List[str] = Field(description="Tournures et mots employés à réutiliser par Emma")
    transitions: List[Transition]
    bonnes_pratiques: List[str] = Field(description="Ce qui rend ce conseiller efficace")
    a_eviter: List[str] = Field(description="Maladresses à ne pas reproduire")

In [ ]:
import anthropic, json
client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"

SYSTEM_ANALYSTE = """Tu es analyste qualité chez SmartSeniors (plateforme de mise en relation familles ↔ EHPAD).
À partir du transcript d'un appel (conseiller ↔ famille en recherche d'établissement), tu extrais une « fiche conseillère »
qui servira à former Emma, l'IA conseillère.

Règles :
- Reformule proprement (corrige les fautes du speech-to-text) SANS jamais inventer d'information absente.
- ANONYMISE dans les verbatims : remplace nom de famille réel, téléphone, email par [ANONYMISÉ].
- Mappe chaque donnée captée au champ de lead correspondant quand c'est possible. Champs disponibles :
  prenom_proche, nom_proche, date_naissance_proche, ville_recherche, code_postal, niveau_autonomie,
  situation_actuelle, budget_mensuel, delai, lien_proche, type_residence, rayon_km,
  contact_prenom, contact_nom, contact_telephone, contact_email.
- Étape "identite" = infos nécessaires pour CRÉER le lead (nom, prénom, date de naissance, CP + ville).
  Étape "solution" = critères de matching (délai, ville + rayon, budget).
- Capture le vocabulaire et les phrases qui créent du lien et de la confiance."""

FICHE_TOOL = {
    "name": "enregistrer_fiche_conseillere",
    "description": "Enregistre la fiche conseillère structurée extraite de l'appel.",
    "input_schema": FicheConseillere.model_json_schema(),
}

def extraire_fiche(transcript: str) -> FicheConseillere:
    resp = client.messages.create(
        model=MODEL,
        max_tokens=16000,
        system=[{"type": "text", "text": SYSTEM_ANALYSTE, "cache_control": {"type": "ephemeral"}}],
        tools=[FICHE_TOOL],
        tool_choice={"type": "tool", "name": "enregistrer_fiche_conseillere"},
        messages=[{"role": "user", "content": [
            {"type": "text", "text": "TRANSCRIPT DE L'APPEL :\n\n" + transcript, "cache_control": {"type": "ephemeral"}},
            {"type": "text", "text": "Analyse ce transcript et remplis la fiche conseillère complète."},
        ]}],
    )
    u = resp.usage
    print(f"📊 in={u.input_tokens}  cache_write={u.cache_creation_input_tokens}  "
          f"cache_read={u.cache_read_input_tokens}  out={u.output_tokens}")
    block = next(b for b in resp.content if b.type == "tool_use")
    return FicheConseillere.model_validate(block.input)

In [ ]:
# Source du transcript : dialogue diarizé > brut Whisper > fichier .txt uploadé
if "transcript_dialogue" in globals():
    transcript = transcript_dialogue
elif "plain" in globals():
    transcript = plain
else:
    up = files.upload()                       # un .txt
    transcript = open(next(iter(up))).read()

fiche = extraire_fiche(transcript)
print(json.dumps(fiche.model_dump(), ensure_ascii=False, indent=2)[:3000])

In [ ]:
# Sauvegarde fiche.json + fiche.md lisible
def fiche_to_md(f):
    L = [f"# Fiche conseillère\n\n**Résumé :** {f.resume_appel}\n", "## Questions de découverte (dans l'ordre)\n"]
    for q in sorted(f.questions_decouverte, key=lambda x: x.ordre):
        ref = f" → `{q.champ_lead}`" if q.champ_lead else ""
        L.append(f"{q.ordre}. **[{q.etape}]** {q.question} — _{q.intention}_{ref}")
    L.append("\n## Données captées → champs de lead\n")
    for d in f.points_donnees:
        L.append(f"- `{d.champ_lead}` = **{d.valeur}** _({d.etape})_ — « {d.verbatim} »")
    L.append("\n## Objections & réponses\n")
    for o in f.objections:
        L.append(f"- **Objection :** {o.objection}\n  - **Réponse ({o.technique}) :** {o.reponse_conseiller}")
    L.append("\n## Phrases d'empathie\n")
    for p in f.phrases_empathie:
        L.append(f"- « {p.phrase} » _({p.contexte})_")
    L.append("\n## Vocabulaire\n- " + "\n- ".join(f.vocabulaire))
    L.append("\n## Transitions\n")
    for t in f.transitions:
        L.append(f"- {t.de} → {t.vers} : « {t.formulation} »")
    L.append("\n## Bonnes pratiques\n- " + "\n- ".join(f.bonnes_pratiques))
    L.append("\n## À éviter\n- " + "\n- ".join(f.a_eviter))
    return "\n".join(L)

open("fiche.json", "w").write(json.dumps(fiche.model_dump(), ensure_ascii=False, indent=2))
open("fiche.md", "w").write(fiche_to_md(fiche))
files.download("fiche.md"); files.download("fiche.json")
print("Fiche sauvegardée ✅")

## Partie C — Plusieurs appels + synthèse en « playbook Emma »

Dépose plusieurs transcripts `.txt` dans le dossier `transcripts/` (panneau Fichiers à gauche) puis lance la boucle :
le préfixe `system + outil` est servi **depuis le cache** dès le 2ᵉ appel (`cache_read` > 0).
La synthèse fusionne toutes les fiches en un playbook prêt à coller dans le prompt d'Emma.

In [ ]:
import glob, os, json
os.makedirs("transcripts", exist_ok=True); os.makedirs("fiches", exist_ok=True)

paths = sorted(glob.glob("transcripts/*.txt"))
fiches = []
for p in paths:
    print("→", p)
    f = extraire_fiche(open(p).read())
    fiches.append(f)
    json.dump(f.model_dump(), open(f"fiches/{os.path.basename(p)}.json", "w"), ensure_ascii=False, indent=2)

if not fiches:
    fiches = [fiche]   # au moins l'appel courant
print(len(fiches), "fiche(s) prête(s) pour la synthèse")

In [ ]:
SYNTHESE_SYS = """Tu es responsable formation chez SmartSeniors. À partir de fiches conseillères issues de vrais appels,
tu produis un PLAYBOOK condensé et actionnable pour Emma, l'IA conseillère. Pas de blabla : des règles et des formulations réutilisables."""

corpus = [f.model_dump() for f in fiches]
prompt = (
    f"Voici {len(corpus)} fiche(s) conseillère(s) en JSON :\n\n"
    + json.dumps(corpus, ensure_ascii=False)
    + "\n\nProduis un playbook **Markdown** prêt à coller dans le prompt système d'Emma, avec ces sections :\n"
      "1. Ordre de découverte optimal (questions types, une par étape)\n"
      "2. Scripts d'objection (objection → réponse)\n"
      "3. Phrases d'empathie réutilisables\n"
      "4. Vocabulaire imposé\n"
      "5. Carte « ce que la famille dit » → « champ de lead » (étape identite puis solution)\n"
      "6. Transitions entre étapes\n"
      "Condense, factuel, directement exploitable."
)

resp = client.messages.create(model=MODEL, max_tokens=16000, system=SYNTHESE_SYS,
                              messages=[{"role": "user", "content": prompt}])
playbook = "".join(b.text for b in resp.content if b.type == "text")
open("emma-playbook.md", "w").write(playbook)
print(playbook[:3000])
files.download("emma-playbook.md")

## Boucler : comment Emma « apprend » à devenir une vraie conseillère

Emma ne fait pas de fine-tuning — elle progresse par **un corpus qui nourrit son prompt** :

```
Appels réels (.m4a)
   └─(Whisper)→ transcripts
        └─(Claude tool use)→ fiches conseillères structurées   → knowledge/fiches/
             └─(Claude synthèse)→ playbook                      → knowledge/emma-playbook.md
                  └─(toi)→ règles + exemples distillés          → pages/functions/api/chat.js (BASE_SYSTEM_PROMPT)
```

À chaque nouvel appel ajouté, le corpus grandit et le playbook s'affine → Emma pose de meilleures questions,
capte mieux les données du lead, et gère mieux les objections.

**Intégrer dans `chat.js`** : reprends les meilleures sections du playbook et ajoute-les au `BASE_SYSTEM_PROMPT`
(ou charge-les comme un bloc « PLAYBOOK » dédié). Garde le prompt court : privilégie quelques formulations fortes
et la carte donnée→champ_lead plutôt que tout recopier.

**RGPD** : ne committe jamais un transcript brut nominatif. Travaille en local depuis `knowledge/transcripts/raw/`
(ignoré par git), et n'archive que les versions anonymisées (`knowledge/transcripts/clean/`) et les fiches.